# 1 Electronic Markets and the Limit Order Book

## 1.1 Electronic Markets

### What is traded

- **Ordinary shares (common stock):** fractional ownership of a company
  - carry voting rights
  - residual claim: paid last in bankruptcy
- **Bonds:** debt paying regular, predetermined coupons plus principal at maturity
  - senior to all equity
- **Preferred stock:** equity ranking below debt but above common stock
  - fixed dividend that *can* be suspended (a bond coupon generally cannot)
  - usually no voting rights
- **Cash & cash equivalents:** money-market accounts, savings deposits, Treasury bills
  - highly liquid, low risk
- **Foreign exchange (FX):** currencies traded in pairs
- **Commodities:** energy, metals, agricultural products
- **Real estate / property:** land and buildings
- **Exchange-traded funds (ETFs):** baskets of assets trading like a single share
- **Mutual funds:** pooled investment vehicles
  - *NAV (net asset value):* total value of the fund's holdings divided by the number of shares, the per-share worth of what the fund actually owns
  - *open-ended*: shares issued/redeemed on demand at NAV. fund grows and shrinks
  - *closed-ended*: fixed share count, trades on an exchange, may sit at premium/discount to NAV
- **Hedge funds:** privately offered (invitation-only), lightly regulated
  - free to use leverage, short-selling, and more aggressive strategies
- **Derivatives:** contracts whose value derives from an underlying
  - *options*: the right, not the obligation, to buy/sell at a set price
  - *futures*: the obligation to buy/sell at a set price on a future date
  - *swaps* (e.g. credit default swaps): exchange of cash flows between parties

## 1.2 Classifying Market Participants
* **Corporations** create shares through an IPO or SSOs
  * for their diverse economic activities
  * shares are claims on a corporation
  * **Point:** corporate managers are participants by increasing or reducing the supply of shares
* **Funds (ETF, mutual)** manage large numbers of financial contracts
  * very active participants and create substantial friction in markets
  * "supply side traders"
  * run either long-horizon or very immediate strategies
* **Proprietary Traders**
  * trade on a (sometimes illusory) advantage
* **Fundamental Traders**
  * have a direct use for the traded asset
  * buy stock for dividends, or buy copper to use it
* **Governments**
  * manage currency
  * issue debt
  * purchase assets for liquidity
  * maintain market stability

### Three classes
1. **Fundamental (or noise, or liquidity) traders**
   * trade for reasons rooted in economic fundamentals outside the exchange
2. **Informed Traders**
   * profit from leveraging information
3. **Market Makers**
   * facilitate exchange in a particular asset
   * exploit skill in executing trades

## 1.3 Trading in Electronic Markets
* Basically two types of orders:
  * **Limit Order (LO):** passive bid/ask for a quantity at a stated price
  * **Market Order (MO):** buy/sell a quantity at any available price
* Orders are managed by a **Matching Engine** and a **Limit Order Book (LOB)**
  * **LOB:** keeps track of incoming and outgoing orders (the resting state of the market)
  * **Matching Engine:** a well-defined algorithm that establishes when a trade can occur, and if so, which criterion selects the orders to be executed
  * In my implementation the matching engine lives in the LOBs  `_match_and_rest`

### Matching is not always first-come-first-served
* Price-time priority (FIFO at a price level) is common but not universal
* **Pro-rata** rule in some money markets: fills split across resting orders in proportion to size
* Some venues grant **market-maker priority** ahead of other orders

### Where trading happens
* **Regulated exchanges:** NASDAQ, NYSE
* **Other electronic venues:** ECNs, dark pools, and similar
* the same asset can trade on many venues at once (fragmentation)

### Order types beyond MO and LO
* Many extended order types exist beyond simple market and limit orders
  * e.g. Fill-Or-Kill (execute fully and immediately or cancel)
  * most are exchange-specific

### Trading is not free

* Trading has a cost, and the cost is not the same for everyone
* **Maker-Taker model:**
  * **Taker** (sends the marketable order, removes liquidity) pays a higher fee
  * **Maker** (resting LO that gets filled, provides liquidity) pays a lower fee or even receives a rebate
* maker/taker split in match engines `Fill` list is relevant to profit

## 1.4 The Limit Order Book

In my LOB both LO and MO start by "walking the book".
* a **Market Order** walks until filled (or the opposite side is empty)
* a **Limit Order** walks only up to its limit price, then rests the remainder as a new resting order if not fully filled
* I do not support re-routing (sending unfilled quantity to another venue)

In [5]:
from hft_lib.lob import LimitOrderBook, Side, OrderType, Order, Fill
book = LimitOrderBook()
for i in range(0, 5):
    book.submit_limit_order(i, 10 * i + 5, Side.BID, 100 - i)
for i in range(0,5):
    book.submit_limit_order(i + 5, 12 * i + 3, Side.ASK, 100 + i + 1)
print(book.display())

Order Book:
Ask: 105 x 51
Ask: 104 x 39
Ask: 103 x 27
Ask: 102 x 15
Ask: 101 x 3
--------------------
Bid: 100 x 5
Bid: 99 x 15
Bid: 98 x 25
Bid: 97 x 35
Bid: 96 x 45


**Example 1: a marketable limit buy that partially fills, then rests.**

Submit a limit order to buy 10 at 101.
The best ask is 101 with only 3 resting,
so the order trades there but nowhere higher and creates a new bid.

* it takes all 3 at 101 and *lifts the offer*, leaving 7 still to buy
* the next ask is 102, above the limit of 101, so it stops walking
* the unfilled 7 *rest* as a new bid at 101

In [6]:
fills = book.submit_limit_order(100, 10, Side.BID, 101)
print("Fills:")
print("\n".join(str(f) for f in fills))
print()
print(book.display())

Fills:
Fill(maker_order_id=5, taker_order_id=100, price=101, qty=3)

Order Book:
Ask: 105 x 51
Ask: 104 x 39
Ask: 103 x 27
Ask: 102 x 15
--------------------
Bid: 101 x 7
Bid: 100 x 5
Bid: 99 x 15
Bid: 98 x 25
Bid: 97 x 35
Bid: 96 x 45


**Example 2: a market sell that walks down the book.**

Submit a market order to sell 30.
A sell MO *hits the bid* and consumes
resting buy orders from the best price downward:

* 5 at 100 (the whole top level)
* 15 at 99 (the whole next level)
* 10 at 98, leaving 15 of that level resting

It fills 30 in total across three price levels, each at a worse price than the
last.

The price impact is the additional cost of the parts of
the order getting executed at worse price levels than the initial best bid.

In this case: $15*1 (99-100) + 10*2 (98-100) = 35$.

In [7]:
# clear the book first
book = LimitOrderBook()
for i in range(0, 5):
    book.submit_limit_order(i, 10 * i + 5, Side.BID, 100 - i)
for i in range(0,5):
    book.submit_limit_order(i + 5, 12 * i + 3, Side.ASK, 100 + i + 1)

fills = book.submit_market_order(101, 30, Side.ASK)
print("Fills:")
print("\n".join(str(f) for f in fills))
print()
print(book.display())

Fills:
Fill(maker_order_id=0, taker_order_id=101, price=100, qty=5)
Fill(maker_order_id=1, taker_order_id=101, price=99, qty=15)
Fill(maker_order_id=2, taker_order_id=101, price=98, qty=10)

Order Book:
Ask: 105 x 51
Ask: 104 x 39
Ask: 103 x 27
Ask: 102 x 15
Ask: 101 x 3
--------------------
Bid: 98 x 15
Bid: 97 x 35
Bid: 96 x 45


### Important concepts

* When a **sell MO** executes against a buy LO it **"hits the bid"**
* When a **buy MO** executes against a sell LO it **"lifts the offer"**

* **Stub quotes:** market makers must quote prices, but may post absurdly low bids or high asks so they do not get matched
  * during the Flash Crash these stub quotes actually got executed
  * can be illustrated with a bid at 1 and an ask at 100000
* **Tick:** the size of the price step
  * US: minimum of 1 cent
  * Europe: between 0.001 and 0.05 euro depending on the price
  * matters for order book size, especially on FPGA where one may need a BRAM entry per price level

### Price proxies

* **Quoted Spread** at time t: $\text{Quoted Spread}_t = P^a_t - P^b_t$
  * if bid equals ask the market is **locked**
  * in my LOB a locked book cannot exist, incoming LOs match against the opposite side first
* **Midprice** at time t: $\text{Midprice}_t = \frac{1}{2}(P^a_t + P^b_t)$
  * often used as a proxy for the price of the asset
* **Microprice** at time t: $\text{Microprice}_t = \frac{V_t^b}{V_t^b + V_t^a} P^a_t + \frac{V_t^a}{V_t^b + V_t^a} P^b_t$
  * with $V_t^a$, $V_t^b$ the volumes at best ask and best bid, and $P$ the corresponding prices
  * a better proxy for the "transaction cost-free" price
  * leans toward the heavier side, indicating buy or sell pressure

In [8]:
# clear the book first
book = LimitOrderBook()
for i in range(0, 5):
    book.submit_limit_order(i, 10 * i + 5, Side.BID, 100 - i)
for i in range(0,5):
    book.submit_limit_order(i + 5, 12 * i + 3, Side.ASK, 100 + i + 1)

print(book.display())
print()
print(f"Best bid: {book.best_bid}")
print(f"Best ask: {book.best_ask}")
print(f"Quoted spread: {book.quoted_spread}")
print(f"Midprice: {book.midprice}")
print(f"Microprice: {book.microprice}")

Order Book:
Ask: 105 x 51
Ask: 104 x 39
Ask: 103 x 27
Ask: 102 x 15
Ask: 101 x 3
--------------------
Bid: 100 x 5
Bid: 99 x 15
Bid: 98 x 25
Bid: 97 x 35
Bid: 96 x 45

Best bid: 100
Best ask: 101
Quoted spread: 1
Midprice: 100.5
Microprice: 100.625
